# Part E for Fall Break Study Session

In [2]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
rng = np.random.default_rng(651)

In [3]:
df = pd.read_csv('/Users/grindhardt/Desktop/Python/Project STAT/wnba_shots_2025.csv')
print(df.shape)
print(df.head(10))
print(df.isna().sum().sort_values(ascending=False).head(10))

(38535, 24)
           GRID_TYPE     GAME_ID  GAME_EVENT_ID  PLAYER_ID     PLAYER_NAME  \
0  Shot Chart Detail  1022500004            185     201886  DeWanna Bonner   
1  Shot Chart Detail  1022500004            325     201886  DeWanna Bonner   
2  Shot Chart Detail  1022500004            349     201886  DeWanna Bonner   
3  Shot Chart Detail  1022500004            374     201886  DeWanna Bonner   
4  Shot Chart Detail  1022500004            488     201886  DeWanna Bonner   
5  Shot Chart Detail  1022500004            510     201886  DeWanna Bonner   
6  Shot Chart Detail  1022500004            525     201886  DeWanna Bonner   
7  Shot Chart Detail  1022500004            570     201886  DeWanna Bonner   
8  Shot Chart Detail  1022500004            572     201886  DeWanna Bonner   
9  Shot Chart Detail  1022500011            167     201886  DeWanna Bonner   

      TEAM_ID      TEAM_NAME  PERIOD  MINUTES_REMAINING  SECONDS_REMAINING  \
0  1611661325  Indiana Fever       2               

In [4]:
OUTCOME = 'SHOT_MADE_FLAG'
GROUP = 'ACTION_TYPE' 
ID_COLS = ['GAME_ID', 'PLAYER_ID', 'TEAM_ID', 'GAME_EVENT_ID']
y = (df[OUTCOME] == 1).to_numpy().astype(int) # a 0/1 outcome
# numeric outcome instead: y = df[OUTCOME].to_numpy

In [5]:
# E3: correlations and PCA on numeric inputs only
num = df.select_dtypes("number").drop(columns=ID_COLS + [OUTCOME], errors='ignore')
num = num.loc[:, num.std() > 0]  # Drop constant columns
R = num.corr()

pairs = [
    (a, b, round(float(R.loc[a, b]), 3)) 
    for i, a in enumerate(R.columns) 
    for b in R.columns[i + 1:] 
    if abs(R.loc[a, b]) > 0.8
]
print(pairs)
Z = ((num - num.mean()) / num.std()).fillna(0)  #standardize; gap -> mean
S = np.linalg.svd(Z.to_numpy(), compute_uv=False)
cum = np.cumsum(S**2) / (S**2).sum()
print(np.argmax(cum >= 0.90) + 1, "components reach 90%")

[('SHOT_DISTANCE', 'LOC_Y', 0.8)]
6 components reach 90%


In [6]:
# E5: the one number in each subgroup, with a bootstrap interval
for level in df[GROUP].dropna().unique():
    ys = y[(df[GROUP] == level).to_numpy()]
    boot = [rng.choice(ys, len(ys), replace=True).mean() for _ in range(10_000)]
    print(level, len(ys), ys.mean().round(4),
            np.percentile(boot, [2.5, 97.5]).round(4))

Running Finger Roll Layup Shot 126 0.7063 [0.627  0.7857]
Jump Shot 11502 0.3504 [0.3416 0.3591]
Running Layup Shot 2097 0.65 [0.6295 0.67  ]
Driving Layup Shot 4733 0.4722 [0.4581 0.4864]
Jump Bank Shot 96 0.5104 [0.4062 0.6146]
Tip Layup Shot 208 0.5192 [0.4519 0.5865]
Fadeaway Jump Shot 413 0.3414 [0.2954 0.3874]
Pullup Jump shot 5306 0.3587 [0.346  0.3715]
Cutting Layup Shot 2160 0.7218 [0.7032 0.7407]
Running Jump Shot 1043 0.3941 [0.3643 0.4228]
Turnaround Jump Shot 951 0.3575 [0.327 0.388]
Floating Jump shot 344 0.3721 [0.3227 0.4244]
Driving Finger Roll Layup Shot 616 0.6055 [0.5666 0.6445]
Step Back Jump shot 1386 0.36 [0.3348 0.3853]
Layup Shot 2140 0.5495 [0.5285 0.5706]
Turnaround Fadeaway shot 542 0.4004 [0.3579 0.441 ]
Driving Floating Jump Shot 1193 0.3445 [0.3185 0.3713]
Putback Layup Shot 1060 0.6019 [0.5726 0.6311]
Alley Oop Layup shot 41 0.561 [0.4146 0.7073]
Driving Hook Shot 161 0.4658 [0.3851 0.5404]
Turnaround Hook Shot 279 0.4337 [0.3763 0.491 ]
Hook Shot 212 0.

In [7]:
missing_summary = pd.DataFrame({
    'Missing Count': df.isna().sum(),
    'Missing Pct (%)': (df.isna().sum() / len(df)) * 100
}).sort_values(by='Missing Count', ascending=False)

print(missing_summary[missing_summary['Missing Count'] > 0])  # Shows only columns with missing values

Empty DataFrame
Columns: [Missing Count, Missing Pct (%)]
Index: []
